In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from sklearn.cluster import SpectralClustering
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

from sklearn.decomposition import PCA

In [ ]:
#Load data
PATH = '/Users/venice/2025-26/ECS111/'
df = pd.read_csv(f'{PATH}IPIP300-SCORES.csv')  # adjust filename to match your download

In [ ]:
# preprocessing
ocean_cols = [
    'openness',
    'conscientiousness',
    'extraversion',
    'agreeableness',
    'neuroticism'
]

df_ocean = df[ocean_cols].dropna()
train_df, test_df = train_test_split(df_ocean, test_size=0.2, random_state=42)

scaler = StandardScaler()
x_train = scaler.fit_transform(train_df)
x_test = scaler.transform(test_df)

In [ ]:
# fit the Spectral Clustering model with different types of similarity measurements
model = SpectralClustering(
    n_clusters=6,
    affinity='rbf', # Gaussian kernel
    gamma=1.0, # controls width: higher = tighter, lower = broader
    assign_labels='kmeans',
    random_state=42
)

In [ ]:
'''
VOID: the dataset is too large for spectral clustering
'''
# train_labels = model.fit_predict(x_train)  # returns array of cluster labels

# knn = KNeighborsClassifier(n_neighbors=5)
# knn.fit(x_train, train_labels)
# test_labels = knn.predict(x_test)

# plt.scatter(x_test, c=test_labels, cmap='viridis', edgecolors='k')
# plt.title("Spectral Clustering on Non-Convex Data")
# plt.xlabel("Feature 1")
# plt.ylabel("Feature 2")

In [ ]:
# hyperparameter tuning (manual test for different gamma values)
from sklearn.metrics import silhouette_score, davies_bouldin_score

gammas = [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]
results = []

df_ocean_sample = df_ocean.sample(n=10000, random_state=42)
train_df, test_df = train_test_split(df_ocean_sample, test_size=0.2, random_state=42)

scaler = StandardScaler()
x_train = scaler.fit_transform(train_df)
x_test = scaler.transform(test_df)

for gamma in gammas:
    model = SpectralClustering(n_clusters=6, affinity='rbf', gamma=gamma, random_state=42)
    labels = model.fit_predict(x_train)
    results.append({
        'gamma': gamma,
        'silhouette': silhouette_score(x_train, labels),
        'davies_bouldin': davies_bouldin_score(x_train, labels)  # lower = better
    })

results_df = pd.DataFrame(results)
print(results_df)

In [ ]:
# kernel died
# subsample to prevent kernel crash
df_ocean_sample = df_ocean.sample(n=10000, random_state=42)
train_df, test_df = train_test_split(df_ocean_sample, test_size=0.2, random_state=42)

scaler = StandardScaler()
x_train = scaler.fit_transform(train_df)
x_test = scaler.transform(test_df)

model = SpectralClustering(
    n_clusters=6,
    affinity='rbf',
    gamma=1.0,
    assign_labels='kmeans',
    random_state=42
)

train_labels = model.fit_predict(x_train)
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(x_train, train_labels)
test_labels = knn.predict(x_test)

# visualize
pca = PCA(n_components=2)
x_test_2d = pca.fit_transform(x_test)

plt.scatter(x_test_2d[:, 0], x_test_2d[:, 1], c=test_labels, cmap='viridis', edgecolors='k', s=20)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Spectral Clustering (test set, PCA projection)')
plt.colorbar(label='Cluster')
plt.savefig("spectral_clusters.png")